In [0]:
dbutils.widgets.text("table", "")        # declare the parameter named "table"
table = dbutils.widgets.get("table")     # read the value the Job passed in


In [0]:

def has_parquet(path):
    for item in dbutils.fs.ls(path):
        if item.isDir():
            if has_parquet(item.path):
                return True
        elif item.path.endswith(".parquet"):
            print("found:", item.path)
            return True
    return False


def bronze_write_stream(source_table):
        if has_parquet(source_table):
                
                bronze_stream = (spark.readStream.format("cloudFiles")
                .option("cloudFiles.format", "parquet")
                .option("cloudFiles.schemaLocation",
                        bronze_schema_destination)   # bookmark
                .load(source_table))       # read from SOURCE
                
                (bronze_stream.writeStream.format("parquet")
                .outputMode("append")
                .option("checkpointLocation",
                        f"{bronze}/_checkpoint/{table}")   # bookmark (DIFFERENT folder)
                .option("path",
                        f"{bronze}/{table}")               # write to BRONZE
                .trigger(availableNow=True)
                .start())

        else:
                print(f"No data exist in source table parquet folder {source_table}")

if __name__ == "__main__":
        source = "abfss://travelsource@databricktraveljournal.dfs.core.windows.net"
        bronze = "abfss://bronze@databricktraveljournal.dfs.core.windows.net"
        bronze_schema_destination = f"{bronze}/_schema/{table}"
        source_table = f"{source}/{table}"
        bronze_write_stream(source_table)
